# Week 3 Laboratory: Single-Qubit Gates and Target-State Design

[![View in GitHub](https://img.shields.io/badge/View%20in-GitHub-black?logo=github)](https://github.com/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_03_QC_M34099_Student.ipynb)

**M34099 Quantum Computing: Level 7 (2026/27)**

**Problem:** Starting from $|0\rangle$, design and justify a circuit that prepares

$$|-i\rangle=\frac{|0\rangle-i|1\rangle}{\sqrt 2}.$$

You will predict gate actions, verify unitarity, test circuit order, compare state fidelity and recommend an efficient circuit. Run the cells in order.


## 1. Install and import the required packages

Run the installation cell once. If Google Colab requests a restart, select **Runtime > Restart session**, then continue from the imports cell.


In [ ]:
# Install Qiskit, Aer and circuit drawing support.
%pip install -q "qiskit[visualization]" qiskit-aer pylatexenc


In [ ]:
# Import NumPy for matrix and vector calculations.
import numpy as np

# Import the Qiskit circuit constructor.
from qiskit import QuantumCircuit

# Import exact statevectors and the fidelity calculation.
from qiskit.quantum_info import Statevector, state_fidelity

# Import the Bloch-sphere plotting function.
from qiskit.visualization import plot_bloch_multivector

# Create the computational basis state |0>.
zero = Statevector.from_label("0")

# Create the target state |-i>.
target = Statevector(np.array([1, -1j], dtype=complex) / np.sqrt(2))

# Display the target statevector.
print("Target state:", target.data)


## 2. Gates, matrices and circuit order

A one-qubit gate is represented by a $2\times2$ unitary matrix. A matrix $U$ is unitary when

$$U^\dagger U=I.$$

Qiskit applies circuit gates from left to right in time. In a matrix calculation, the rightmost matrix acts first. Therefore, a circuit that applies $H$ and then $Z$ has the combined matrix $ZH$.

The target state $|-i\rangle$ lies on the negative $y$ direction of the Bloch sphere.


## 3. Task 1: Predict a gate action using a matrix

Calculate $X|0\rangle$ by hand, then run the code to verify the result. Compare the predicted vector with the Qiskit statevector.


In [ ]:
# Define the Pauli-X matrix.
X = np.array([[0, 1], [1, 0]], dtype=complex)

# Define the column vector for |0>.
zero_vector = np.array([1, 0], dtype=complex)

# Multiply X by |0> to calculate the prediction.
predicted = X @ zero_vector

# Display the matrix prediction.
print("Prediction:", predicted)

# Create a one-qubit circuit.
qc_x = QuantumCircuit(1)

# Apply the X gate to qubit 0.
qc_x.x(0)

# Evolve |0> through the circuit.
actual = zero.evolve(qc_x)

# Display the Qiskit statevector.
print("Qiskit state:", actual.data)

# Label the circuit output.
print("Circuit for X|0>:")

# Display the circuit diagram.
display(qc_x.draw("mpl", style="iqp", fold=-1))


### Interpretation

The X gate swaps the two computational-basis amplitudes. It maps $|0\rangle=[1,0]^T$ to $|1\rangle=[0,1]^T$.


## 4. Task 2: Check unitarity

Calculate $U^\dagger U$ for the Hadamard and phase gates. Each valid closed-system quantum gate should produce the identity matrix.


In [ ]:
# Define the Hadamard matrix.
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

# Define the phase-gate matrix.
S = np.array([[1, 0], [0, 1j]], dtype=complex)

# Define a reusable function that calculates U-dagger times U.
def unitarity_check(U):
    # Calculate the conjugate transpose and multiply it by U.
    result = U.conj().T @ U

    # Return the calculated matrix.
    return result

# Calculate the unitarity check for H.
h_check = unitarity_check(H)

# Calculate the unitarity check for S.
s_check = unitarity_check(S)

# Display the result for H.
print("H-dagger H =\n", h_check)

# Display the result for S.
print("S-dagger S =\n", s_check)


## 5. Task 3: Test circuit order

Compare a circuit that applies H then Z with a circuit that applies Z then H. Predict both outputs from $|0\rangle$ before running the code.


In [ ]:
# Create the circuit that applies H and then Z.
qc_hz = QuantumCircuit(1)

# Apply H first.
qc_hz.h(0)

# Apply Z second.
qc_hz.z(0)

# Create the circuit that applies Z and then H.
qc_zh = QuantumCircuit(1)

# Apply Z first.
qc_zh.z(0)

# Apply H second.
qc_zh.h(0)

# Evolve |0> through the H-then-Z circuit.
state_hz = zero.evolve(qc_hz)

# Evolve |0> through the Z-then-H circuit.
state_zh = zero.evolve(qc_zh)

# Display the H-then-Z result.
print("H then Z:", state_hz.data)

# Display the Z-then-H result.
print("Z then H:", state_zh.data)

# Label the H-then-Z circuit.
print("Circuit 1: H then Z")

# Display the H-then-Z circuit separately.
display(qc_hz.draw("mpl", style="iqp", fold=-1))

# Label the Z-then-H circuit.
print("Circuit 2: Z then H")

# Display the Z-then-H circuit separately.
display(qc_zh.draw("mpl", style="iqp", fold=-1))


### Interpretation

H followed by Z prepares $|-\rangle$. Z followed by H prepares $|+\rangle$ because Z does not change the initial $|0\rangle$. The different results show that H and Z do not commute.


## 6. Task 4: Prepare the target state

Circuit A uses H followed by $S^\dagger$. Calculate its fidelity with $|-i\rangle$. Fidelity 1 means the states match up to global phase.


In [ ]:
# Create Circuit A.
circuit_a = QuantumCircuit(1)

# Apply H to create |+>.
circuit_a.h(0)

# Apply S-dagger to create |-i>.
circuit_a.sdg(0)

# Evolve |0> through Circuit A.
state_a = zero.evolve(circuit_a)

# Calculate fidelity with the target state.
fidelity_a = state_fidelity(state_a, target)

# Calculate the circuit depth.
depth_a = circuit_a.depth()

# Calculate the total gate count.
gates_a = sum(circuit_a.count_ops().values())

# Display the final statevector.
print("Circuit A state:", state_a.data)

# Display the fidelity.
print(f"Fidelity with target: {fidelity_a:.6f}")

# Display the circuit depth and gate count.
print("Depth:", depth_a, "Gate count:", gates_a)

# Label the circuit output.
print("Circuit A: H then S-dagger")

# Display the circuit diagram.
display(circuit_a.draw("mpl", style="iqp", fold=-1))

# Display the final state on the Bloch sphere.
display(plot_bloch_multivector(state_a, title="Circuit A final state: |-i>"))


### Visual interpretation

The Bloch vector should point along the negative $y$ axis. This direction represents the relative phase $-i$ between the $|0\rangle$ and $|1\rangle$ amplitudes.


## 7. Task 5: Compare an alternative circuit

Circuit B may use only H and S. Since $S^\dagger=S^3$, use H followed by three S gates. Compare fidelity, gate count and depth with Circuit A.


In [ ]:
# Create Circuit B.
circuit_b = QuantumCircuit(1)

# Apply H to create |+>.
circuit_b.h(0)

# Apply the first S gate.
circuit_b.s(0)

# Apply the second S gate.
circuit_b.s(0)

# Apply the third S gate so that S cubed equals S-dagger.
circuit_b.s(0)

# Evolve |0> through Circuit B.
state_b = zero.evolve(circuit_b)

# Calculate fidelity with the target state.
fidelity_b = state_fidelity(state_b, target)

# Calculate the circuit depth.
depth_b = circuit_b.depth()

# Calculate the total gate count.
gates_b = sum(circuit_b.count_ops().values())

# Store the comparison in a readable dictionary.
comparison = {
    "A": {"fidelity": fidelity_a, "depth": depth_a, "gates": gates_a},
    "B": {"fidelity": fidelity_b, "depth": depth_b, "gates": gates_b},
}

# Label the comparison output.
print("Circuit comparison")

# Display the metrics for Circuit A.
print(f"A: fidelity={fidelity_a:.6f}, depth={depth_a}, gates={gates_a}")

# Display the metrics for Circuit B.
print(f"B: fidelity={fidelity_b:.6f}, depth={depth_b}, gates={gates_b}")

# Label the alternative circuit output.
print("Circuit B: H then S then S then S")

# Display the alternative circuit.
display(circuit_b.draw("mpl", style="iqp", fold=-1))


## 8. PBL evidence record

Use the Week 3 PBL activity to make your recommendation. Your evidence should:

1. state the gates in time order and give the corresponding matrix product;
2. include a hand calculation or intermediate states;
3. report fidelity, gate count and depth for both circuits;
4. explain why global phase does not affect the conclusion; and
5. identify one limitation of ideal statevector evidence.

**Assessment connection:** Gate calculations and circuit interpretation rehearse LO2 CBT questions. The topic exploration supports preparation for the formative LO3 report milestone.


## 9. Review questions

1. Why does the written matrix product reverse the time order of the circuit gates?
2. What does $U^\dagger U=I$ demonstrate?
3. Why do H then Z and Z then H produce different states from $|0\rangle$?
4. Why is Circuit A preferable to Circuit B in the ideal comparison?
5. What practical factors could change the comparison on real hardware?


## References

- [IBM Quantum Learning: Single-system gates](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/single-systems/single-system-gates)
- [Qiskit QuantumCircuit API](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit)
- [Qiskit Statevector API](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.Statevector)

Accessed 5 October 2026.
